In [1]:
import torch
import torch.nn as nn
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import warnings
from tqdm import tqdm
import os
import json
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence


# Ignore specific warnings
warnings.filterwarnings("ignore")

class MyGRU(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super(MyGRU,self).__init__()
        self.gru = nn.GRU(in_dim, hidden_dim, batch_first=True)

    def forward(self, x, batch):
        # x: [total_num_nodes, feature_dim]
        # batch: [total_num_nodes] → 图编号

        # Step 1: 按图划分节点特征
        graphs = []
        lengths = []

        num_graphs = batch.max().item() + 1
        for i in range(num_graphs):
            node_feats = x[batch == i]              # shape: [num_nodes_i, feature_dim]
           
            sorted_nodes = node_feats
            graphs.append(sorted_nodes)
            lengths.append(sorted_nodes.size(0))

        # Step 2: pad 成相同长度序列 → [batch_size, max_seq_len, feature_dim]
        padded_seqs = pad_sequence(graphs, batch_first=True)   # 自动 padding 0
        lengths = torch.tensor(lengths)

        # Step 3: pack 序列送入 GRU
        packed_input = pack_padded_sequence(padded_seqs, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, h_n = self.gru(packed_input)

        # h_n: [1, batch_size, hidden_dim] → 最后时间步的隐藏状态
        # 我们返回 squeeze 掉层数
        return h_n.squeeze(0)   # shape: [batch_size, hidden_dim]


class MultiScaleGAT(torch.nn.Module):
    def __init__(self, input_size, hidden_channels):
        super(MultiScaleGAT, self).__init__()
        torch.manual_seed(12345)
        # GAT layers for spatial feature extraction
        self.conv1 = GCNConv(input_size, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.conv4 = GCNConv(hidden_channels, hidden_channels)
        self.conv5 = GCNConv(hidden_channels, hidden_channels)
        
        # Self-attention for multi-scale feature fusion
        self.attention = nn.MultiheadAttention(hidden_channels, num_heads=1)
    
    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings from GAT layers
        x1 = self.conv1(x, edge_index).relu()
        x2 = self.conv2(x1, edge_index).relu()
        x3 = self.conv3(x2, edge_index).relu()
        x4 = self.conv4(x3, edge_index).relu()
        x5 = self.conv5(x4, edge_index)

        # 2. Multi-scale feature fusion
        # Global mean pooling for each GAT layer
        x1_pool = global_mean_pool(x1, batch)  # [batch_size, hidden_channels]
        x2_pool = global_mean_pool(x2, batch)  # [batch_size, hidden_channels]
        x3_pool = global_mean_pool(x3, batch)  # [batch_size, hidden_channels]
        x4_pool = global_mean_pool(x4, batch)  # [batch_size, hidden_channels]
        x5_pool = global_mean_pool(x5, batch)  # [batch_size, hidden_channels]

        # Stack pooled features for self-attention
        # Shape: [3, batch_size, hidden_channels]
        multi_scale_features = torch.stack([x1_pool, x2_pool, x3_pool, x4_pool, x5_pool], dim=0)
        
        # Self-attention to fuse multi-scale features
        # Input to MultiheadAttention expects [seq_len, batch_size, hidden_channels]
        attn_output, _ = self.attention(multi_scale_features, multi_scale_features, multi_scale_features)
        # Sum or average over sequence dimension to get fused spatial feature
        spatial_feature = attn_output.mean(dim=0)  # [batch_size, hidden_channels]

        return spatial_feature

class FUSION(torch.nn.Module):
    def __init__(self, input_size, hidden_channels):
        super(FUSION, self).__init__()
        self.gat =  MultiScaleGAT(input_size, hidden_channels)
        self.gru =  MyGRU(input_size, hidden_channels)
        # Final linear layer
        self.lin = Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        spatial_feature = self.gat(x, edge_index, batch)
        h_n = self.gru(x, batch)
        fusion_feature = spatial_feature + h_n
        fusion_feature = F.dropout(fusion_feature, p=0.2, training=self.training)
        output = self.lin(fusion_feature)
        return  output


In [2]:

def load_json_folder(folder_path):
    data_array = []

    # Iterate through files in the folder
    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)

        # Check if the file is a JSON file
        if filename.endswith(".json"):
            # Read and load the JSON file
            with open(file_path, 'r') as json_file:
                data = json.load(json_file)

            # Append the loaded data to the array
            data_array.append(data)


    return data_array
def convert_to_one_hot(label_tensor, num_classes):
    """
    Convert a label tensor to its one-hot encoded representation.

    Args:
    - label_tensor (torch.Tensor): Tensor containing the labels.
    - num_classes (int): Number of classes.

    Returns:
    - torch.Tensor: One-hot encoded tensor.
    """
    # Initialize the one-hot encoded tensor
    one_hot_tensor = torch.zeros(len(label_tensor), num_classes)

    # Fill in the one-hot encoded tensor
    one_hot_tensor[range(len(label_tensor)), label_tensor] = 1

    return one_hot_tensor
def train(model, optimizer, criterion, loader, device):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model.train()
    total_loss = 0
    y_true = []
    y_pred = []

    for data in loader:  # Iterate in batches over the train dataset.
        # Ensure data is on the correct device and type
        data.x = data.x.to(device, dtype=torch.float32)  # Convert to float32 and ensure on GPU
        data.edge_index = data.edge_index.to(device, dtype=torch.long)  # Ensure edge_index is long and on GPU
        data.y = data.y.to(device, dtype=torch.long)  # Ensure y is long and on GPU
        data.batch = data.batch.to(device) if data.batch is not None else None  # Handle batch index for batched graphs
        # batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
        
        # Forward pass
        out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass
        loss = criterion(out, data.y)  # Compute loss (CrossEntropyLoss expects long labels, not one-hot)
        loss.backward()  # Derive gradients
        optimizer.step()  # Update parameters based on gradients
        optimizer.zero_grad()  # Clear gradients

        # Compute predictions for metrics
        pred = out.argmax(dim=1)  # Get predicted class
        y_true.extend(data.y.cpu().tolist())  # Move to CPU for sklearn metrics
        y_pred.extend(pred.cpu().tolist())  # Move to CPU for sklearn metrics
        total_loss += loss.item()  # Accumulate loss (scale by batch size)

    # Calculate metrics
    average_loss = total_loss / len(loader)  # Average loss per graph
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return average_loss, accuracy, precision, recall, f1, model
    
def test(model, loader):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model.eval()
    y_true = []
    y_pred = []
    correct = 0

    for data in loader:  # Iterate in batches over the training/test dataset.
        data.edge_index = data.edge_index.to(torch.long)
        data.x = data.x.to(model.parameters().__next__().dtype)
        # batch = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        out = model(data.x, data.edge_index, data.batch)
        pred = out.argmax(dim=1)  # Use the class with the highest probability.
        y_true.extend(data.y.tolist())
        y_pred.extend(pred.tolist())
        # correct += int((pred == data.y).sum())  # Check against ground-truth labels.

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)  # or 'micro' or 'weighted'
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    # Compute accuracy
    # accuracy = correct / len(loader.dataset)

    return accuracy, precision, recall, f1

In [3]:

# Assuming train, test, load_json_folder, and FUSION are defined elsewhere
# from your_module import train, test, load_json_folder, FUSION

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load training data
train_data = load_json_folder("data/reentrancy_train")
Train_data = []
for data in train_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)
    inputsize = x.shape[1]
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)
    Train_data.append(Data(x=x, edge_index=edges, y=y))

# Load test data
Test_data = []
test_data = load_json_folder("data/reentrancy_test")
for data in test_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)
    Test_data.append(Data(x=x, edge_index=edges, y=y))

# Create DataLoader instances
train_loader = DataLoader(Train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(Test_data, batch_size=32, shuffle=False)

# Initialize model, criterion, and optimizer
model = FUSION(hidden_channels=128, input_size=inputsize).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Early stopping parameters
patience = 5  # Number of epochs to wait for improvement
best_accuracy = 0.0
epochs_no_improve = 0
best_model_path = 'model/reentrancy_model_best.pth'

# Training and testing loop
for epoch in range(200):
    # Training
    model.train()
    train_loss, train_accuracy, train_precision, train_recall, train_f1, model = train(model, optimizer, criterion, train_loader, device)

    # Testing
    model.eval()
    test_accuracy, test_precision, test_recall, test_f1 = test(model, test_loader)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | "
              f"Train Precision: {train_precision:.4f} | Train Recall: {train_recall:.4f} | Train F1: {train_f1:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f} | Test Precision: {test_precision:.4f} | "
              f"Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}")

    # Early stopping logic
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        epochs_no_improve = 0
        # Save the best model
        torch.save(model.state_dict(), best_model_path)
        print(f"New best test accuracy: {best_accuracy:.4f}, model saved to {best_model_path}")
    else:
        epochs_no_improve += 1
        # print(f"No improvement in test accuracy. Epochs without improvement: {epochs_no_improve}")

    # # Check for early stopping
    # if epochs_no_improve >= patience:
    #     # print(f"Early stopping triggered after {epoch + 1} epochs. No improvement in test accuracy for {patience} epochs.")
    #     break

# Load and evaluate the best model
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
final_accuracy, final_precision, final_recall, final_f1 = test(model, test_loader)
print(f"Final Best Model | Accuracy: {final_accuracy:.4f} | Precision: {final_precision:.4f} | "
      f"Recall: {final_recall:.4f} | F1 Score: {final_f1:.4f}")
print(f"Best model saved at: {best_model_path}")

Using device: cuda


100%|██████████| 54/54 [00:00<00:00, 1338.18it/s]


New best test accuracy: 0.7222, model saved to model/reentrancy_model_best.pth
New best test accuracy: 0.8519, model saved to model/reentrancy_model_best.pth
New best test accuracy: 0.8704, model saved to model/reentrancy_model_best.pth
Epoch 10 | Train Loss: 0.2479 | Train Accuracy: 0.8920 | Train Precision: 0.8839 | Train Recall: 0.8316 | Train F1: 0.8527
Test Accuracy: 0.8333 | Test Precision: 0.7143 | Test Recall: 0.6667 | Test F1: 0.6897
Epoch 20 | Train Loss: 0.1104 | Train Accuracy: 0.9531 | Train Precision: 0.9498 | Train Recall: 0.9290 | Train F1: 0.9387
Test Accuracy: 0.8333 | Test Precision: 0.7143 | Test Recall: 0.6667 | Test F1: 0.6897
Epoch 30 | Train Loss: 0.0837 | Train Accuracy: 0.9577 | Train Precision: 0.9655 | Train Recall: 0.9266 | Train F1: 0.9438
Test Accuracy: 0.8333 | Test Precision: 0.7143 | Test Recall: 0.6667 | Test F1: 0.6897
Epoch 40 | Train Loss: 0.0193 | Train Accuracy: 0.9953 | Train Precision: 0.9968 | Train Recall: 0.9912 | Train F1: 0.9940
Test Accur